In [ ]:
!pip install autogluon
!pip install optuna 


^C


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, f1_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

import optuna
from autogluon.tabular import TabularPredictor


In [ ]:
health_df = pd.read_csv("healthcare-dataset-stroke-data.csv")
health_df.head()
health_df.info()
print("Пропуски:\n", health_df.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB
Пропуски:
 id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married         

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
health_df['bmi'] = imputer.fit_transform(health_df[['bmi']])

health_df['gender'] = health_df['gender'].replace('Other', 'Male')

le = LabelEncoder()
for col in ['gender','ever_married','Residence_type']:
    health_df[col] = le.fit_transform(health_df[col])

health_df = pd.get_dummies(health_df, columns=['work_type', 'smoking_status'], drop_first=True)

if 'id' in health_df.columns:
    health_df = health_df.drop(columns=['id'])


In [ ]:
X = health_df.drop('stroke', axis=1)
y = health_df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
scaler = StandardScaler()

num_cols = ['age', 'avg_glucose_level', 'bmi']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train_np = X_train.values.astype('float32')
X_test_np = X_test.values.astype('float32')
y_train_np = y_train.values.astype('int32')
y_test_np = y_test.values.astype('int32')


Optuna

In [ ]:
def create_model(trial):
    n_units = trial.suggest_int("n_units", 16, 128)
    layers_num = trial.suggest_int("layers", 1, 4)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)

    model = Sequential()
    model.add(Dense(n_units, activation="relu", input_shape=(X_train.shape[1],)))

    for _ in range(layers_num):
        model.add(Dense(n_units, activation="relu"))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


def objective(trial):
    model = create_model(trial)

    history = model.fit(
        X_train_np, y_train_np,
        epochs=trial.suggest_int("epochs", 10, 50),
        batch_size=trial.suggest_int("batch", 16, 128),
        validation_data=(X_test_np, y_test_np),
        verbose=0
    )

    val_acc = history.history["val_accuracy"][-1]
    return val_acc


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Лучшие параметры:", study.best_params)
print("Лучшая точность:", study.best_value)


[I 2025-12-08 07:55:23,770] A new study created in memory with name: no-name-475a4b6b-ea5c-460d-8961-eed3bc1d1aaf
[I 2025-12-08 07:55:38,532] Trial 0 finished with value: 0.9403131008148193 and parameters: {'n_units': 38, 'layers': 4, 'lr': 0.008999777883537792, 'epochs': 26, 'batch': 94}. Best is trial 0 with value: 0.9403131008148193.
[I 2025-12-08 07:56:00,492] Trial 1 finished with value: 0.951076328754425 and parameters: {'n_units': 101, 'layers': 1, 'lr': 0.0010784490701401309, 'epochs': 36, 'batch': 117}. Best is trial 1 with value: 0.951076328754425.
[I 2025-12-08 07:56:26,046] Trial 2 finished with value: 0.9246575236320496 and parameters: {'n_units': 81, 'layers': 1, 'lr': 0.009464213642659147, 'epochs': 34, 'batch': 55}. Best is trial 1 with value: 0.951076328754425.
[I 2025-12-08 07:56:44,055] Trial 3 finished with value: 0.951076328754425 and parameters: {'n_units': 104, 'layers': 3, 'lr': 1.5914434451051754e-05, 'epochs': 22, 'batch': 70}. Best is trial 1 with value: 0.95

Лучшие параметры: {'n_units': 108, 'layers': 3, 'lr': 0.00020058455199505355, 'epochs': 35, 'batch': 124}
Лучшая точность: 0.9530332684516907


In [ ]:
best_params = study.best_params
best_model = create_model(optuna.trial.FixedTrial(best_params))

best_model.fit(
    X_train_np, y_train_np,
    epochs=best_params["epochs"],
    batch_size=best_params["batch"],
    verbose=1
)

pred = (best_model.predict(X_test_np) > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test_np, pred))
print(classification_report(y_test_np, pred))


Epoch 1/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9551 - loss: 0.5382
Epoch 2/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9542 - loss: 0.2587
Epoch 3/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9519 - loss: 0.2083
Epoch 4/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9563 - loss: 0.1695
Epoch 5/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9538 - loss: 0.1604
Epoch 6/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9516 - loss: 0.1609
Epoch 7/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9527 - loss: 0.1494
Epoch 8/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9550 - loss: 0.1480
Epoch 9/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9493 - loss: 0.1591
Epoch 10/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9484 - loss: 0.1646
Epoch 11/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9576 - loss: 0.1377
Epoch 12/35
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9516

Autogluon

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor

data = pd.read_csv("healthcare-dataset-stroke-data.csv")

train_data = TabularDataset(data)

predictor = TabularPredictor(
    label="stroke",
).fit(
    train_data=train_data,
    time_limit=300,
    presets="medium"
)


No path specified. Models will be saved in: "AutogluonModels\ag-20251208_030657"
Preset alias specified: 'medium' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.0
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          2
Memory Avail:       1.04 GB / 7.78 GB (13.3%)
Disk Space Avail:   12.21 GB / 115.64 GB (10.6%)
Presets specified: ['medium']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "c:\Users\Админ\Desktop\ЭЦП физ\notete\Lab12\AutogluonModels\ag-20251208_030657"
Train Data Rows:    5110
Train Data Columns: 11
Label Column:       stroke
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenera

In [ ]:
predictor.leaderboard(silent=True)


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBMXT,0.951076,accuracy,0.003986,14.438755,0.003986,14.438755,1,True,1
1,LightGBMLarge,0.951076,accuracy,0.003986,3.151416,0.003986,3.151416,1,True,11
2,CatBoost,0.951076,accuracy,0.003986,16.091517,0.003986,16.091517,1,True,5
3,LightGBM,0.951076,accuracy,0.004983,2.651238,0.004983,2.651238,1,True,2
4,XGBoost,0.951076,accuracy,0.009967,1.523001,0.009967,1.523001,1,True,9
5,NeuralNetTorch,0.951076,accuracy,0.021132,22.709203,0.021132,22.709203,1,True,10
6,NeuralNetFastAI,0.951076,accuracy,0.067775,13.013989,0.067775,13.013989,1,True,8
7,WeightedEnsemble_L2,0.951076,accuracy,0.068772,13.178175,0.000997,0.164186,2,True,12
8,RandomForestEntr,0.951076,accuracy,0.083387,2.290090,0.083387,2.290090,1,True,4
9,ExtraTreesEntr,0.951076,accuracy,0.086895,1.792464,0.086895,1.792464,1,True,7
